In [3]:
import pandas as pd
import numpy as np

# Task
Implement Gradient Boosting Machines (GBM) from scratch for a regression task. This involves providing an algorithm introduction, implementing a loss function and its negative gradient, creating a decision stump/tree as a base learner, building the core GBM greedy function approximation, developing a prediction method, and demonstrating its usage on a synthetic dataset.

## Algorithm Introduction

### Subtask:
Provide a brief introduction to Gradient Boosting Machines (GBM) as a greedy function approximation method, explaining the core concepts outlined in Friedman's paper.


## Introduction to Gradient Boosting Machines (GBM)

Gradient Boosting Machines (GBM) represent a powerful and widely used ensemble learning technique for both regression and classification tasks. It operates as a **greedy function approximation method**, where the model is built sequentially by combining multiple simpler, weaker models (often decision trees).

The core idea behind GBM, significantly popularized by Friedman's work, is to iteratively improve the model's predictions by focusing on the errors of the previous iterations. In each step, a new weak learner is trained to predict the **negative gradient of the loss function** with respect to the current ensemble's predictions. Essentially, instead of directly fitting the new model to the residuals (the difference between the observed and predicted values), it fits to the *direction of the steepest descent* of the loss function. This process allows the model to progressively correct the mistakes made by the prior learners, leading to a highly accurate and robust predictive model.

## Loss Function and Negative Gradient

### Subtask:
Implement a loss function (e.g., Mean Squared Error for regression) and its negative gradient. This is crucial as GBM iteratively fits base learners to these negative gradients.


**Reasoning**:
Implement the `mse_loss` and `mse_negative_gradient` functions as described in the instructions, including comments for clarity.



In [5]:
def mse_loss(y_true, y_pred):
    """
    Calculates the Mean Squared Error (MSE) between true and predicted values.

    Args:
        y_true (array-like): True target values.
        y_pred (array-like): Predicted target values.

    Returns:
        float: The Mean Squared Error.
    """
    return np.mean((y_true - y_pred)**2)

def mse_negative_gradient(y_true, y_pred):
    """
    Calculates the negative gradient of the Mean Squared Error with respect to the predictions.
    For MSE, the negative gradient is (y_true - y_pred).

    Args:
        y_true (array-like): True target values.
        y_pred (array-like): Predicted target values.

    Returns:
        array-like: The negative gradient for each prediction.
    """
    return y_true - y_pred

print("MSE loss and negative gradient functions are defined.")

MSE loss and negative gradient functions are defined.


## Base Learner Implementation (Decision Stump/Tree)

### Subtask:
Create a simple decision tree (or decision stump) from scratch to serve as the weak learner. This will involve implementing splitting criteria and prediction for individual trees.


**Reasoning**:
I will start by defining the `Node` class which is a fundamental building block for the decision tree, and then define the `DecisionTreeRegressor` class with its initializer, including parameters for controlling tree growth such as `max_depth` and `min_samples_split`.



In [6]:
class Node:
    """
    Represents a node in the decision tree.
    """
    def __init__(self, feature_index=None, threshold=None, left=None, right=None, value=None):
        # For decision node
        self.feature_index = feature_index  # Index of the feature to split on
        self.threshold = threshold          # Threshold value for the split
        self.left = left                    # Left child node
        self.right = right                  # Right child node

        # For leaf node
        self.value = value                  # Predicted value if it's a leaf node

class DecisionTreeRegressor:
    """
    A simple Decision Tree Regressor implementation.
    """
    def __init__(self, max_depth=1, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None # The root node of the tree

    print("Node and DecisionTreeRegressor classes defined with initializer.")

Node and DecisionTreeRegressor classes defined with initializer.


**Reasoning**:
Now that the basic class structure is defined, I will implement the `_best_split` private method inside the `DecisionTreeRegressor` class. This method will find the best feature and threshold to split the data, based on minimizing the Mean Squared Error (MSE).



In [7]:
class Node:
    """
    Represents a node in the decision tree.
    """
    def __init__(self, feature_index=None, threshold=None, left=None, right=None, value=None):
        # For decision node
        self.feature_index = feature_index  # Index of the feature to split on
        self.threshold = threshold          # Threshold value for the split
        self.left = left                    # Left child node
        self.right = right                  # Right child node

        # For leaf node
        self.value = value                  # Predicted value if it's a leaf node

class DecisionTreeRegressor:
    """
    A simple Decision Tree Regressor implementation.
    """
    def __init__(self, max_depth=1, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None # The root node of the tree

    def _best_split(self, X, y):
        """
        Finds the best feature and threshold to split the data.
        It aims to minimize the Mean Squared Error (MSE) of the split.
        """
        n_samples, n_features = X.shape
        if n_samples <= self.min_samples_split:
            return None, None

        # Calculate MSE of the parent node (before splitting)
        parent_mse = np.mean((y - np.mean(y))**2)
        best_gain = 0 # Maximize information gain or minimize weighted MSE of children
        best_feature_index = None
        best_threshold = None

        for feature_index in range(n_features):
            feature_values = X[:, feature_index]
            possible_thresholds = np.unique(feature_values)

            for threshold in possible_thresholds:
                # Split data based on the current feature and threshold
                left_indices = np.where(feature_values <= threshold)[0]
                right_indices = np.where(feature_values > threshold)[0]

                if len(left_indices) == 0 or len(right_indices) == 0:
                    continue

                # Calculate weighted MSE of children
                y_left, y_right = y[left_indices], y[right_indices]

                mse_left = np.mean((y_left - np.mean(y_left))**2) if len(y_left) > 0 else 0
                mse_right = np.mean((y_right - np.mean(y_right))**2) if len(y_right) > 0 else 0

                n_left, n_right = len(y_left), len(y_right)
                weighted_mse_children = (n_left * mse_left + n_right * mse_right) / n_samples

                # Calculate reduction in MSE (gain)
                gain = parent_mse - weighted_mse_children

                if gain > best_gain:
                    best_gain = gain
                    best_feature_index = feature_index
                    best_threshold = threshold

        return best_feature_index, best_threshold

print("DecisionTreeRegressor class updated with _best_split method.")


DecisionTreeRegressor class updated with _best_split method.


**Reasoning**:
I will implement the `_split` private method inside the `DecisionTreeRegressor` class. This method will take the dataset, a feature index, and a threshold, and return the left and right subsets of the data based on the split.



In [8]:
class Node:
    """
    Represents a node in the decision tree.
    """
    def __init__(self, feature_index=None, threshold=None, left=None, right=None, value=None):
        # For decision node
        self.feature_index = feature_index  # Index of the feature to split on
        self.threshold = threshold          # Threshold value for the split
        self.left = left                    # Left child node
        self.right = right                  # Right child node

        # For leaf node
        self.value = value                  # Predicted value if it's a leaf node

class DecisionTreeRegressor:
    """
    A simple Decision Tree Regressor implementation.
    """
    def __init__(self, max_depth=1, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None # The root node of the tree

    def _best_split(self, X, y):
        """
        Finds the best feature and threshold to split the data.
        It aims to minimize the Mean Squared Error (MSE) of the split.
        """
        n_samples, n_features = X.shape
        if n_samples <= self.min_samples_split:
            return None, None

        # Calculate MSE of the parent node (before splitting)
        parent_mse = np.mean((y - np.mean(y))**2)
        best_gain = 0 # Maximize information gain or minimize weighted MSE of children
        best_feature_index = None
        best_threshold = None

        for feature_index in range(n_features):
            feature_values = X[:, feature_index]
            possible_thresholds = np.unique(feature_values)

            for threshold in possible_thresholds:
                # Split data based on the current feature and threshold
                left_indices = np.where(feature_values <= threshold)[0]
                right_indices = np.where(feature_values > threshold)[0]

                if len(left_indices) == 0 or len(right_indices) == 0:
                    continue

                # Calculate weighted MSE of children
                y_left, y_right = y[left_indices], y[right_indices]

                mse_left = np.mean((y_left - np.mean(y_left))**2) if len(y_left) > 0 else 0
                mse_right = np.mean((y_right - np.mean(y_right))**2) if len(y_right) > 0 else 0

                n_left, n_right = len(y_left), len(y_right)
                weighted_mse_children = (n_left * mse_left + n_right * mse_right) / n_samples

                # Calculate reduction in MSE (gain)
                gain = parent_mse - weighted_mse_children

                if gain > best_gain:
                    best_gain = gain
                    best_feature_index = feature_index
                    best_threshold = threshold

        return best_feature_index, best_threshold

    def _split(self, X, y, feature_index, threshold):
        """
        Splits the dataset into left and right subsets based on a feature and threshold.
        """
        left_indices = np.where(X[:, feature_index] <= threshold)[0]
        right_indices = np.where(X[:, feature_index] > threshold)[0]

        X_left, y_left = X[left_indices], y[left_indices]
        X_right, y_right = X[right_indices], y[right_indices]

        return X_left, y_left, X_right, y_right

print("DecisionTreeRegressor class updated with _split method.")

DecisionTreeRegressor class updated with _split method.


**Reasoning**:
Now, I will implement the `_leaf_value` private method inside the `DecisionTreeRegressor` class. This method calculates the prediction for a leaf node, which for regression, is the mean of the target values in that node.



In [9]:
class Node:
    """
    Represents a node in the decision tree.
    """
    def __init__(self, feature_index=None, threshold=None, left=None, right=None, value=None):
        # For decision node
        self.feature_index = feature_index  # Index of the feature to split on
        self.threshold = threshold          # Threshold value for the split
        self.left = left                    # Left child node
        self.right = right                  # Right child node

        # For leaf node
        self.value = value                  # Predicted value if it's a leaf node

class DecisionTreeRegressor:
    """
    A simple Decision Tree Regressor implementation.
    """
    def __init__(self, max_depth=1, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None # The root node of the tree

    def _best_split(self, X, y):
        """
        Finds the best feature and threshold to split the data.
        It aims to minimize the Mean Squared Error (MSE) of the split.
        """
        n_samples, n_features = X.shape
        if n_samples <= self.min_samples_split:
            return None, None

        # Calculate MSE of the parent node (before splitting)
        parent_mse = np.mean((y - np.mean(y))**2)
        best_gain = 0 # Maximize information gain or minimize weighted MSE of children
        best_feature_index = None
        best_threshold = None

        for feature_index in range(n_features):
            feature_values = X[:, feature_index]
            possible_thresholds = np.unique(feature_values)

            for threshold in possible_thresholds:
                # Split data based on the current feature and threshold
                left_indices = np.where(feature_values <= threshold)[0]
                right_indices = np.where(feature_values > threshold)[0]

                if len(left_indices) == 0 or len(right_indices) == 0:
                    continue

                # Calculate weighted MSE of children
                y_left, y_right = y[left_indices], y[right_indices]

                mse_left = np.mean((y_left - np.mean(y_left))**2) if len(y_left) > 0 else 0
                mse_right = np.mean((y_right - np.mean(y_right))**2) if len(y_right) > 0 else 0

                n_left, n_right = len(y_left), len(y_right)
                weighted_mse_children = (n_left * mse_left + n_right * mse_right) / n_samples

                # Calculate reduction in MSE (gain)
                gain = parent_mse - weighted_mse_children

                if gain > best_gain:
                    best_gain = gain
                    best_feature_index = feature_index
                    best_threshold = threshold

        return best_feature_index, best_threshold

    def _split(self, X, y, feature_index, threshold):
        """
        Splits the dataset into left and right subsets based on a feature and threshold.
        """
        left_indices = np.where(X[:, feature_index] <= threshold)[0]
        right_indices = np.where(X[:, feature_index] > threshold)[0]

        X_left, y_left = X[left_indices], y[left_indices]
        X_right, y_right = X[right_indices], y[right_indices]

        return X_left, y_left, X_right, y_right

    def _leaf_value(self, y):
        """
        Calculates the prediction for a leaf node (mean of target values).
        """
        if len(y) == 0:
            return 0
        return np.mean(y)

print("DecisionTreeRegressor class updated with _leaf_value method.")

DecisionTreeRegressor class updated with _leaf_value method.


**Reasoning**:
Now, I will implement the private recursive method, `_grow_tree`, inside the `DecisionTreeRegressor` class. This method is responsible for recursively building the decision tree by finding the best splits and creating child nodes until the maximum depth or minimum samples split criteria are met.



In [10]:
class Node:
    """
    Represents a node in the decision tree.
    """
    def __init__(self, feature_index=None, threshold=None, left=None, right=None, value=None):
        # For decision node
        self.feature_index = feature_index  # Index of the feature to split on
        self.threshold = threshold          # Threshold value for the split
        self.left = left                    # Left child node
        self.right = right                  # Right child node

        # For leaf node
        self.value = value                  # Predicted value if it's a leaf node

class DecisionTreeRegressor:
    """
    A simple Decision Tree Regressor implementation.
    """
    def __init__(self, max_depth=1, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None # The root node of the tree

    def _best_split(self, X, y):
        """
        Finds the best feature and threshold to split the data.
        It aims to minimize the Mean Squared Error (MSE) of the split.
        """
        n_samples, n_features = X.shape
        if n_samples <= self.min_samples_split:
            return None, None

        # Calculate MSE of the parent node (before splitting)
        parent_mse = np.mean((y - np.mean(y))**2)
        best_gain = 0 # Maximize information gain or minimize weighted MSE of children
        best_feature_index = None
        best_threshold = None

        for feature_index in range(n_features):
            feature_values = X[:, feature_index]
            possible_thresholds = np.unique(feature_values)

            for threshold in possible_thresholds:
                # Split data based on the current feature and threshold
                left_indices = np.where(feature_values <= threshold)[0]
                right_indices = np.where(feature_values > threshold)[0]

                if len(left_indices) == 0 or len(right_indices) == 0:
                    continue

                # Calculate weighted MSE of children
                y_left, y_right = y[left_indices], y[right_indices]

                mse_left = np.mean((y_left - np.mean(y_left))**2) if len(y_left) > 0 else 0
                mse_right = np.mean((y_right - np.mean(y_right))**2) if len(y_right) > 0 else 0

                n_left, n_right = len(y_left), len(y_right)
                weighted_mse_children = (n_left * mse_left + n_right * mse_right) / n_samples

                # Calculate reduction in MSE (gain)
                gain = parent_mse - weighted_mse_children

                if gain > best_gain:
                    best_gain = gain
                    best_feature_index = feature_index
                    best_threshold = threshold

        return best_feature_index, best_threshold

    def _split(self, X, y, feature_index, threshold):
        """
        Splits the dataset into left and right subsets based on a feature and threshold.
        """
        left_indices = np.where(X[:, feature_index] <= threshold)[0]
        right_indices = np.where(X[:, feature_index] > threshold)[0]

        X_left, y_left = X[left_indices], y[left_indices]
        X_right, y_right = X[right_indices], y[right_indices]

        return X_left, y_left, X_right, y_right

    def _leaf_value(self, y):
        """
        Calculates the prediction for a leaf node (mean of target values).
        """
        if len(y) == 0:
            return 0
        return np.mean(y)

    def _grow_tree(self, X, y, depth):
        """
        Recursively builds the decision tree.
        """
        n_samples, n_features = X.shape

        # Stop splitting if max_depth is reached or min_samples_split is not met
        if depth >= self.max_depth or n_samples < self.min_samples_split:
            return Node(value=self._leaf_value(y))

        best_feature, best_threshold = self._best_split(X, y)

        # If no good split found, make it a leaf node
        if best_feature is None:
            return Node(value=self._leaf_value(y))

        # Split the data
        X_left, y_left, X_right, y_right = self._split(X, y, best_feature, best_threshold)

        # Recursively build left and right subtrees
        left_child = self._grow_tree(X_left, y_left, depth + 1)
        right_child = self._grow_tree(X_right, y_right, depth + 1)

        return Node(feature_index=best_feature, threshold=best_threshold, left=left_child, right=right_child)

print("DecisionTreeRegressor class updated with _grow_tree method.")

DecisionTreeRegressor class updated with _grow_tree method.


**Reasoning**:
Now, I will implement the public `fit` method for `DecisionTreeRegressor`. This method will take training data `X` and `y`, initialize the root of the tree, and call the `_grow_tree` method to build the tree.



In [11]:
class Node:
    """
    Represents a node in the decision tree.
    """
    def __init__(self, feature_index=None, threshold=None, left=None, right=None, value=None):
        # For decision node
        self.feature_index = feature_index  # Index of the feature to split on
        self.threshold = threshold          # Threshold value for the split
        self.left = left                    # Left child node
        self.right = right                  # Right child node

        # For leaf node
        self.value = value                  # Predicted value if it's a leaf node

class DecisionTreeRegressor:
    """
    A simple Decision Tree Regressor implementation.
    """
    def __init__(self, max_depth=1, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None # The root node of the tree

    def _best_split(self, X, y):
        """
        Finds the best feature and threshold to split the data.
        It aims to minimize the Mean Squared Error (MSE) of the split.
        """
        n_samples, n_features = X.shape
        if n_samples <= self.min_samples_split:
            return None, None

        # Calculate MSE of the parent node (before splitting)
        parent_mse = np.mean((y - np.mean(y))**2)
        best_gain = 0 # Maximize information gain or minimize weighted MSE of children
        best_feature_index = None
        best_threshold = None

        for feature_index in range(n_features):
            feature_values = X[:, feature_index]
            possible_thresholds = np.unique(feature_values)

            for threshold in possible_thresholds:
                # Split data based on the current feature and threshold
                left_indices = np.where(feature_values <= threshold)[0]
                right_indices = np.where(feature_values > threshold)[0]

                if len(left_indices) == 0 or len(right_indices) == 0:
                    continue

                # Calculate weighted MSE of children
                y_left, y_right = y[left_indices], y[right_indices]

                mse_left = np.mean((y_left - np.mean(y_left))**2) if len(y_left) > 0 else 0
                mse_right = np.mean((y_right - np.mean(y_right))**2) if len(y_right) > 0 else 0

                n_left, n_right = len(y_left), len(y_right)
                weighted_mse_children = (n_left * mse_left + n_right * mse_right) / n_samples

                # Calculate reduction in MSE (gain)
                gain = parent_mse - weighted_mse_children

                if gain > best_gain:
                    best_gain = gain
                    best_feature_index = feature_index
                    best_threshold = threshold

        return best_feature_index, best_threshold

    def _split(self, X, y, feature_index, threshold):
        """
        Splits the dataset into left and right subsets based on a feature and threshold.
        """
        left_indices = np.where(X[:, feature_index] <= threshold)[0]
        right_indices = np.where(X[:, feature_index] > threshold)[0]

        X_left, y_left = X[left_indices], y[left_indices]
        X_right, y_right = X[right_indices], y[right_indices]

        return X_left, y_left, X_right, y_right

    def _leaf_value(self, y):
        """
        Calculates the prediction for a leaf node (mean of target values).
        """
        if len(y) == 0:
            return 0
        return np.mean(y)

    def _grow_tree(self, X, y, depth):
        """
        Recursively builds the decision tree.
        """
        n_samples, n_features = X.shape

        # Stop splitting if max_depth is reached or min_samples_split is not met
        if depth >= self.max_depth or n_samples < self.min_samples_split:
            return Node(value=self._leaf_value(y))

        best_feature, best_threshold = self._best_split(X, y)

        # If no good split found, make it a leaf node
        if best_feature is None:
            return Node(value=self._leaf_value(y))

        # Split the data
        X_left, y_left, X_right, y_right = self._split(X, y, best_feature, best_threshold)

        # Recursively build left and right subtrees
        left_child = self._grow_tree(X_left, y_left, depth + 1)
        right_child = self._grow_tree(X_right, y_right, depth + 1)

        return Node(feature_index=best_feature, threshold=best_threshold, left=left_child, right=right_child)

    def fit(self, X, y):
        """
        Builds the decision tree regressor from the training data.
        """
        self.root = self._grow_tree(X, y, depth=0)

print("DecisionTreeRegressor class updated with fit method.")

DecisionTreeRegressor class updated with fit method.


**Reasoning**:
Now, I will implement the private recursive method, `_predict_tree`, inside the `DecisionTreeRegressor` class. This method will traverse the built tree for a single input sample `x` and return its predicted value by following the decision rules from the root down to a leaf node.



In [12]:
class Node:
    """
    Represents a node in the decision tree.
    """
    def __init__(self, feature_index=None, threshold=None, left=None, right=None, value=None):
        # For decision node
        self.feature_index = feature_index  # Index of the feature to split on
        self.threshold = threshold          # Threshold value for the split
        self.left = left                    # Left child node
        self.right = right                  # Right child node

        # For leaf node
        self.value = value                  # Predicted value if it's a leaf node

class DecisionTreeRegressor:
    """
    A simple Decision Tree Regressor implementation.
    """
    def __init__(self, max_depth=1, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None # The root node of the tree

    def _best_split(self, X, y):
        """
        Finds the best feature and threshold to split the data.
        It aims to minimize the Mean Squared Error (MSE) of the split.
        """
        n_samples, n_features = X.shape
        if n_samples <= self.min_samples_split:
            return None, None

        # Calculate MSE of the parent node (before splitting)
        parent_mse = np.mean((y - np.mean(y))**2)
        best_gain = 0 # Maximize information gain or minimize weighted MSE of children
        best_feature_index = None
        best_threshold = None

        for feature_index in range(n_features):
            feature_values = X[:, feature_index]
            possible_thresholds = np.unique(feature_values)

            for threshold in possible_thresholds:
                # Split data based on the current feature and threshold
                left_indices = np.where(feature_values <= threshold)[0]
                right_indices = np.where(feature_values > threshold)[0]

                if len(left_indices) == 0 or len(right_indices) == 0:
                    continue

                # Calculate weighted MSE of children
                y_left, y_right = y[left_indices], y[right_indices]

                mse_left = np.mean((y_left - np.mean(y_left))**2) if len(y_left) > 0 else 0
                mse_right = np.mean((y_right - np.mean(y_right))**2) if len(y_right) > 0 else 0

                n_left, n_right = len(y_left), len(y_right)
                weighted_mse_children = (n_left * mse_left + n_right * mse_right) / n_samples

                # Calculate reduction in MSE (gain)
                gain = parent_mse - weighted_mse_children

                if gain > best_gain:
                    best_gain = gain
                    best_feature_index = feature_index
                    best_threshold = threshold

        return best_feature_index, best_threshold

    def _split(self, X, y, feature_index, threshold):
        """
        Splits the dataset into left and right subsets based on a feature and threshold.
        """
        left_indices = np.where(X[:, feature_index] <= threshold)[0]
        right_indices = np.where(X[:, feature_index] > threshold)[0]

        X_left, y_left = X[left_indices], y[left_indices]
        X_right, y_right = X[right_indices], y[right_indices]

        return X_left, y_left, X_right, y_right

    def _leaf_value(self, y):
        """
        Calculates the prediction for a leaf node (mean of target values).
        """
        if len(y) == 0:
            return 0
        return np.mean(y)

    def _grow_tree(self, X, y, depth):
        """
        Recursively builds the decision tree.
        """
        n_samples, n_features = X.shape

        # Stop splitting if max_depth is reached or min_samples_split is not met
        if depth >= self.max_depth or n_samples < self.min_samples_split:
            return Node(value=self._leaf_value(y))

        best_feature, best_threshold = self._best_split(X, y)

        # If no good split found, make it a leaf node
        if best_feature is None:
            return Node(value=self._leaf_value(y))

        # Split the data
        X_left, y_left, X_right, y_right = self._split(X, y, best_feature, best_threshold)

        # Recursively build left and right subtrees
        left_child = self._grow_tree(X_left, y_left, depth + 1)
        right_child = self._grow_tree(X_right, y_right, depth + 1)

        return Node(feature_index=best_feature, threshold=best_threshold, left=left_child, right=right_child)

    def fit(self, X, y):
        """
        Builds the decision tree regressor from the training data.
        """
        self.root = self._grow_tree(X, y, depth=0)

    def _predict_tree(self, x, node):
        """
        Recursively traverses the tree to predict the value for a single sample x.
        """
        if node.value is not None:
            return node.value

        if x[node.feature_index] <= node.threshold:
            return self._predict_tree(x, node.left)
        else:
            return self._predict_tree(x, node.right)

print("DecisionTreeRegressor class updated with _predict_tree method.")

DecisionTreeRegressor class updated with _predict_tree method.


**Reasoning**:
Now, I will implement the public `predict` method for `DecisionTreeRegressor`. This method will take an array of samples `X` and use the `_predict_tree` method to return an array of predictions for the entire input dataset.



In [13]:
class Node:
    """
    Represents a node in the decision tree.
    """
    def __init__(self, feature_index=None, threshold=None, left=None, right=None, value=None):
        # For decision node
        self.feature_index = feature_index  # Index of the feature to split on
        self.threshold = threshold          # Threshold value for the split
        self.left = left                    # Left child node
        self.right = right                  # Right child node

        # For leaf node
        self.value = value                  # Predicted value if it's a leaf node

class DecisionTreeRegressor:
    """
    A simple Decision Tree Regressor implementation.
    """
    def __init__(self, max_depth=1, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None # The root node of the tree

    def _best_split(self, X, y):
        """
        Finds the best feature and threshold to split the data.
        It aims to minimize the Mean Squared Error (MSE) of the split.
        """
        n_samples, n_features = X.shape
        if n_samples <= self.min_samples_split:
            return None, None

        # Calculate MSE of the parent node (before splitting)
        parent_mse = np.mean((y - np.mean(y))**2)
        best_gain = 0 # Maximize information gain or minimize weighted MSE of children
        best_feature_index = None
        best_threshold = None

        for feature_index in range(n_features):
            feature_values = X[:, feature_index]
            possible_thresholds = np.unique(feature_values)

            for threshold in possible_thresholds:
                # Split data based on the current feature and threshold
                left_indices = np.where(feature_values <= threshold)[0]
                right_indices = np.where(feature_values > threshold)[0]

                if len(left_indices) == 0 or len(right_indices) == 0:
                    continue

                # Calculate weighted MSE of children
                y_left, y_right = y[left_indices], y[right_indices]

                mse_left = np.mean((y_left - np.mean(y_left))**2) if len(y_left) > 0 else 0
                mse_right = np.mean((y_right - np.mean(y_right))**2) if len(y_right) > 0 else 0

                n_left, n_right = len(y_left), len(y_right)
                weighted_mse_children = (n_left * mse_left + n_right * mse_right) / n_samples

                # Calculate reduction in MSE (gain)
                gain = parent_mse - weighted_mse_children

                if gain > best_gain:
                    best_gain = gain
                    best_feature_index = feature_index
                    best_threshold = threshold

        return best_feature_index, best_threshold

    def _split(self, X, y, feature_index, threshold):
        """
        Splits the dataset into left and right subsets based on a feature and threshold.
        """
        left_indices = np.where(X[:, feature_index] <= threshold)[0]
        right_indices = np.where(X[:, feature_index] > threshold)[0]

        X_left, y_left = X[left_indices], y[left_indices]
        X_right, y_right = X[right_indices], y[right_indices]

        return X_left, y_left, X_right, y_right

    def _leaf_value(self, y):
        """
        Calculates the prediction for a leaf node (mean of target values).
        """
        if len(y) == 0:
            return 0
        return np.mean(y)

    def _grow_tree(self, X, y, depth):
        """
        Recursively builds the decision tree.
        """
        n_samples, n_features = X.shape

        # Stop splitting if max_depth is reached or min_samples_split is not met
        if depth >= self.max_depth or n_samples < self.min_samples_split:
            return Node(value=self._leaf_value(y))

        best_feature, best_threshold = self._best_split(X, y)

        # If no good split found, make it a leaf node
        if best_feature is None:
            return Node(value=self._leaf_value(y))

        # Split the data
        X_left, y_left, X_right, y_right = self._split(X, y, best_feature, best_threshold)

        # Recursively build left and right subtrees
        left_child = self._grow_tree(X_left, y_left, depth + 1)
        right_child = self._grow_tree(X_right, y_right, depth + 1)

        return Node(feature_index=best_feature, threshold=best_threshold, left=left_child, right=right_child)

    def fit(self, X, y):
        """
        Builds the decision tree regressor from the training data.
        """
        self.root = self._grow_tree(X, y, depth=0)

    def _predict_tree(self, x, node):
        """
        Recursively traverses the tree to predict the value for a single sample x.
        """
        if node.value is not None:
            return node.value

        if x[node.feature_index] <= node.threshold:
            return self._predict_tree(x, node.left)
        else:
            return self._predict_tree(x, node.right)

    def predict(self, X):
        """
        Predicts values for an array of samples X.
        """
        if self.root is None:
            raise Exception("Tree not fitted. Call fit() first.")

        return np.array([self._predict_tree(x, self.root) for x in X])

print("DecisionTreeRegressor class updated with predict method.")

DecisionTreeRegressor class updated with predict method.


## Greedy Function Approximation (GBM) Core

### Subtask:
Implement the main boosting loop, where the model iteratively trains base learners on the negative gradients of the loss function and updates the ensemble.


**Reasoning**:
I will start by defining the `GradientBoostingRegressor` class and its `__init__` method, which will initialize key parameters like the number of estimators, learning rate, tree depth, and minimum samples per split, along with an empty list to store the trained weak learners.



In [14]:
class GradientBoostingRegressor:
    """
    A Gradient Boosting Regressor implementation from scratch.
    """
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3, min_samples_split=2):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.trees = []  # List to store individual decision trees
        self.initial_prediction = None # Initial prediction (mean of y)

print("GradientBoostingRegressor class defined with initializer.")

GradientBoostingRegressor class defined with initializer.


**Reasoning**:
Now, I will implement the `fit` method for the `GradientBoostingRegressor` class. This method will initialize the first prediction with the mean of the target variable `y` and then iterate through the boosting process, training new `DecisionTreeRegressor` models on the negative gradients (residuals) and updating the ensemble prediction in each step.



In [15]:
class GradientBoostingRegressor:
    """
    A Gradient Boosting Regressor implementation from scratch.
    """
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3, min_samples_split=2):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.trees = []  # List to store individual decision trees
        self.initial_prediction = None # Initial prediction (mean of y)

    def fit(self, X, y):
        """
        Fits the Gradient Boosting Regressor model to the training data.
        """
        # 1. Initialize the first prediction with the mean of y
        self.initial_prediction = np.mean(y)
        current_predictions = np.full_like(y, self.initial_prediction, dtype=float)

        # 2. Iterate through n_estimators
        for _ in range(self.n_estimators):
            # a. Calculate the negative gradients (residuals)
            residuals = mse_negative_gradient(y, current_predictions)

            # b. Instantiate a DecisionTreeRegressor as the base learner
            tree = DecisionTreeRegressor(max_depth=self.max_depth, min_samples_split=self.min_samples_split)

            # c. Train the current DecisionTreeRegressor on X and the residuals
            tree.fit(X, residuals)

            # d. Add the trained base learner to the list
            self.trees.append(tree)

            # e. Update the ensemble's prediction
            # Predictions from the tree are estimates of the negative gradient
            tree_predictions = tree.predict(X)
            current_predictions += self.learning_rate * tree_predictions

print("GradientBoostingRegressor class updated with fit method.")

GradientBoostingRegressor class updated with fit method.


**Reasoning**:
Now, I will implement the `predict` method for the `GradientBoostingRegressor` class. This method will take new input data `X` and generate predictions by combining the `initial_prediction` with the weighted predictions from all the trained `DecisionTreeRegressor` base learners.



In [16]:
class GradientBoostingRegressor:
    """
    A Gradient Boosting Regressor implementation from scratch.
    """
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3, min_samples_split=2):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.trees = []  # List to store individual decision trees
        self.initial_prediction = None # Initial prediction (mean of y)

    def fit(self, X, y):
        """
        Fits the Gradient Boosting Regressor model to the training data.
        """
        # 1. Initialize the first prediction with the mean of y
        self.initial_prediction = np.mean(y)
        current_predictions = np.full_like(y, self.initial_prediction, dtype=float)

        # 2. Iterate through n_estimators
        for _ in range(self.n_estimators):
            # a. Calculate the negative gradients (residuals)
            residuals = mse_negative_gradient(y, current_predictions)

            # b. Instantiate a DecisionTreeRegressor as the base learner
            tree = DecisionTreeRegressor(max_depth=self.max_depth, min_samples_split=self.min_samples_split)

            # c. Train the current DecisionTreeRegressor on X and the residuals
            tree.fit(X, residuals)

            # d. Add the trained base learner to the list
            self.trees.append(tree)

            # e. Update the ensemble's prediction
            # Predictions from the tree are estimates of the negative gradient
            tree_predictions = tree.predict(X)
            current_predictions += self.learning_rate * tree_predictions

    def predict(self, X):
        """
        Predicts target values for new input data X.
        """
        if self.initial_prediction is None:
            raise Exception("Model not fitted. Call fit() first.")

        # Start with the initial prediction
        predictions = np.full(X.shape[0], self.initial_prediction)

        # Add the weighted predictions from each tree
        for tree in self.trees:
            predictions += self.learning_rate * tree.predict(X)

        return predictions

print("GradientBoostingRegressor class updated with predict method.")

GradientBoostingRegressor class updated with predict method.


## Demonstrate Usage on a Synthetic Dataset

### Subtask:
Generate a synthetic dataset, instantiate the `GradientBoostingRegressor`, fit the model to the data, make predictions, and evaluate its performance. This step will bring together all the previously implemented components.

## Demonstrate Usage on a Synthetic Dataset

### Subtask:
Generate a synthetic dataset, instantiate the `GradientBoostingRegressor`, fit the model to the data, make predictions, and evaluate its performance. This step will bring together all the previously implemented components.

**Reasoning**:
Now I will generate a synthetic dataset to test the GradientBoostingRegressor. This dataset will be simple, with a clear non-linear relationship to observe how well the GBM can approximate it.



In [17]:
from sklearn.model_selection import train_test_split

# Generate synthetic data
np.random.seed(42)
X = np.random.rand(100, 1) * 10  # 100 samples, 1 feature, values from 0 to 10
y = np.sin(X).flatten() + np.random.randn(100) * 0.5  # y = sin(x) + noise

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Synthetic dataset generated and split into training and testing sets.")
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

Synthetic dataset generated and split into training and testing sets.
X_train shape: (80, 1), y_train shape: (80,)
X_test shape: (20, 1), y_test shape: (20,)


**Reasoning**:
Now that the synthetic dataset is prepared, I will instantiate the `GradientBoostingRegressor` with appropriate parameters and then fit it to the training data (`X_train`, `y_train`).



In [18]:
gbr = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=2, min_samples_split=5)
gbr.fit(X_train, y_train)

print("GradientBoostingRegressor instantiated and fitted to the training data.")

GradientBoostingRegressor instantiated and fitted to the training data.


**Reasoning**:
Now that the model is fitted, I will make predictions on the test set (`X_test`) using the trained `GradientBoostingRegressor` and store these predictions.



In [19]:
y_pred = gbr.predict(X_test)

print("Predictions made on the test set.")

Predictions made on the test set.


**Reasoning**:
With predictions available, the final step for demonstrating usage is to evaluate the model's performance on the test set. I will calculate the Mean Squared Error (MSE) between the true target values (`y_test`) and the predicted values (`y_pred`).



In [20]:
from sklearn.metrics import mean_squared_error

mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error on test set: {mse:.4f}")

Mean Squared Error on test set: 0.1798


## Summary:

### Data Analysis Key Findings

*   **GBM Introduction:** An introduction to Gradient Boosting Machines (GBM) was provided, explaining it as a greedy function approximation method that iteratively trains weak learners on the negative gradient of the loss function, building upon Friedman's work.
*   **Loss Function and Negative Gradient:** The Mean Squared Error (MSE) loss function and its negative gradient (residuals, \(y_{true} - y_{pred}\)) were successfully implemented, which are crucial for guiding the boosting process.
*   **Base Learner (Decision Tree) Implementation:** A `DecisionTreeRegressor` class was implemented from scratch. This included:
    *   A `Node` class for tree structure.
    *   A `_best_split` method to find optimal split points by minimizing MSE.
    *   `_split` and `_leaf_value` helper methods.
    *   A recursive `_grow_tree` method for tree construction, incorporating `max_depth` and `min_samples_split` stopping criteria.
    *   `fit` and `predict` methods for training and inference, respectively.
*   **Gradient Boosting Regressor Core Implementation:** The `GradientBoostingRegressor` class was built, encompassing:
    *   Initialization of parameters like `n_estimators`, `learning_rate`, `max_depth`, and `min_samples_split`.
    *   A `fit` method that initializes predictions with the mean of the target, then iteratively trains `n_estimators` decision trees on the negative gradients (residuals), and updates the ensemble predictions with a scaled contribution (\(\text{learning\_rate} \times \text{tree\_prediction}\)).
    *   A `predict` method that combines the initial prediction with the weighted predictions from all trained base learners.
*   **Demonstration on Synthetic Dataset:**
    *   A synthetic dataset was generated (\(y = \sin(X) + \text{noise}\)) and split into training and testing sets.
    *   The implemented `GradientBoostingRegressor` was trained on this data using 100 estimators, a learning rate of 0.1, and a maximum tree depth of 2.
    *   Predictions were successfully made on the test set.
    *   The model achieved a Mean Squared Error (MSE) of approximately **0.1798** on the test set, demonstrating its ability to learn and generalize on the synthetic data.

### Insights or Next Steps

*   The successful implementation from scratch of GBM components (loss, base learner, boosting loop) provides a solid foundational understanding of the algorithm's mechanics.
*   Further enhancements could include implementing other loss functions (e.g., Huber loss for robustness), incorporating regularization techniques (e.g., shrinkage, subsampling), or exploring different base learners beyond simple decision trees to improve model performance and generalization.
